In [1]:
# Env Setup
import os
import sys

rootpath = "/home/huahua/Projects/transformers-3.3.1"
sys.path.insert(0, os.path.join(rootpath + "/" + "src"))

# Logging
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(rootpath + "/tmp/rag_sequence.log", mode="a", encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))

console_handler = logging.StreamHandler()
logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")

logger.addHandler(file_handler)
logger.addHandler(console_handler)

In [2]:
# Global variable
import argparse
import torch

args = argparse.Namespace()
args.model_name_or_path = rootpath + "/" + "models/rag-sequence-nq"
args.model_type = "confidential_rag_sequence"
args.evaluation_set = rootpath + "/" + "examples/rag/output/biencoder-nq-dev.questions"
args.gold_data_path = rootpath + "/" + "examples/rag/output/biencoder-nq-dev.ans"

# args.evaluation_set = (
#     "examples/rag/output/biencoder-nq-dev-small.questions"
# )
# args.gold_data_path = "examples/rag/output/biencoder-nq-dev-small.ans"

args.predictions_path = rootpath + "/" + "examples/rag/output/e2e_preds.txt"
args.gold_data_mode = "ans"
args.eval_mode = "e2e"
args.n_docs = 10
args.index_name = None
args.index_path = None
args.k = 1
args.eval_all_checkpoints = False
args.eval_batch_size = 4
args.print_predictions = True
args.recalculate = True
args.num_beams = 4
args.min_length = 1
args.max_length = 50
args.print_docs = False
args.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args.numConfidential = 5
args.generationMode = "parallelSummary"

In [150]:
# ConfidentialRagRetriever
import random
from typing import List

import numpy as np
from transformers.tokenization_utils_base import BatchEncoding
from transformers.retrieval_rag import RagRetriever


class ConfidentialRagRetriever(RagRetriever):
    def __init__(
        self,
        config,
        question_encoder_tokenizer,
        generator_tokenizer,
        index=None,
    ):
        self.config = config
        self.n_docs = config.n_docs
        self.batch_size = config.retrieval_batch_size

        self.generator_tokenizer = generator_tokenizer
        self.question_encoder_tokenizer = question_encoder_tokenizer

        self.index = index

    def concatConfidentialDocs(self, inputIds, questionIputIds, confidentialDocs, return_tensors=None):
        def cat_input_and_doc(doc_title, doc_text, question=""):
            if doc_title.startswith('"'):
                doc_title = doc_title[1:]
            if doc_title.endswith('"'):
                doc_title = doc_title[:-1]
            out = (" " + doc_title + self.config.title_sep + doc_text + self.config.doc_sep + question).replace("  ", " ")
            return out

        questionInputStrings = self.question_encoder_tokenizer.batch_decode(questionIputIds, skip_special_tokens=True)

        confidentialRagInputStrings = [
            cat_input_and_doc(
                doc_title=confidentialDocs[i]["title"][j],
                doc_text=confidentialDocs[i]["text"][j],
                question=questionInputStrings[i] if len(questionInputStrings) > 0 else "",
            )
            for i in range(len(confidentialDocs))
            for j in range(len(confidentialDocs[0]["title"]))
        ]
        contextualized_inputs = self.generator_tokenizer.batch_encode_plus(
            confidentialRagInputStrings,
            max_length=self.config.max_combined_length,
            return_tensors=return_tensors,
            padding="max_length",
            truncation=True,
        )
        return BatchEncoding(
            {
                "input_ids": contextualized_inputs["input_ids"],
                "attention_mask": contextualized_inputs["attention_mask"],
            },
            tensor_type=return_tensors,
        )

    def postprocess_docs(self, docs, input_strings, prefix, n_docs, return_tensors=None):
        def cat_input_and_doc(doc_title, doc_text, input_string, prefix):
            # TODO(Patrick): if we train more RAG models, I want to put the input first to take advantage of effortless truncation
            # TODO(piktus): better handling of truncation
            if doc_title.startswith('"'):
                doc_title = doc_title[1:]
            if doc_title.endswith('"'):
                doc_title = doc_title[:-1]
            if prefix is None:
                prefix = ""
            out = (prefix + doc_title + self.config.title_sep + doc_text + self.config.doc_sep + input_string).replace(
                "  ", " "
            )
            return out

        ctxInputStrings = [
            cat_input_and_doc(
                docs[i]["title"][j],
                docs[i]["text"][j],
                input_strings[i],
                prefix,
            )
            for i in range(len(docs))
            for j in range(n_docs)
        ]

        contextualized_inputs = self.generator_tokenizer.batch_encode_plus(
            ctxInputStrings,
            max_length=self.config.max_combined_length,
            return_tensors=return_tensors,
            padding="max_length",
            truncation=True,
        )

        return (
            contextualized_inputs["input_ids"],
            contextualized_inputs["attention_mask"],
        )

    def __call__(
        self,
        question_input_ids: List[List[int]],
        question_hidden_states: np.ndarray,
        prefix=None,
        n_docs=None,
        numConfidential=0,
        return_tensors=None,
    ) -> BatchEncoding:
        n_docs = n_docs if n_docs is not None else self.n_docs
        prefix = prefix if prefix is not None else self.config.generator.prefix
        retrieved_doc_embeds, doc_ids, docs = self.retrieve(question_hidden_states, n_docs + numConfidential)

        input_strings = self.question_encoder_tokenizer.batch_decode(question_input_ids, skip_special_tokens=True)

        normalDocs = []
        confidentialDocs = []
        for doc in docs:
            title2text = list(zip(doc["title"], doc["text"]))
            random.shuffle(title2text)
            normalDocs.append(
                {
                    "title": [item[0] for item in title2text[:n_docs]],
                    "text": [item[1] for item in title2text[:n_docs]],
                }
            )
            confidentialDocs.append(
                {
                    "title": [item[0] for item in title2text[n_docs:]],
                    "text": [item[1] for item in title2text[n_docs:]],
                }
            )

        context_input_ids, context_attention_mask = self.postprocess_docs(
            normalDocs, input_strings, prefix, n_docs, return_tensors=return_tensors
        )
        return (
            BatchEncoding(
                {
                    "context_input_ids": context_input_ids,
                    "context_attention_mask": context_attention_mask,
                    "retrieved_doc_embeds": retrieved_doc_embeds[:, :n_docs],
                    "doc_ids": doc_ids[:, :n_docs],
                },
                tensor_type=return_tensors,
            ),
            confidentialDocs,
        )

In [4]:
# ConfidentialRagModel
from typing import Optional, Tuple, Union
from transformers.modeling_rag import RagModel, RetrievAugLMOutput
from transformers import PretrainedConfig, PreTrainedModel
import torch


class ConfidentialRagModel(RagModel):
    def __init__(
        self,
        config: Optional[PretrainedConfig] = None,
        question_encoder: Optional[PreTrainedModel] = None,
        generator: Optional[PreTrainedModel] = None,
        retriever: Optional = None,  # or maybe just use a `set_retriever(...)` method # type: ignore
        **kwargs,
    ):
        super().__init__(
            config=config,
            question_encoder=question_encoder,
            generator=generator,
            retriever=retriever,
            kwargs=kwargs,
        )
        self.numConfidential = 0

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        encoder_outputs: Optional[Tuple[Tuple[torch.FloatTensor]]] = None,
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.BoolTensor] = None,
        past_key_values: Optional[Tuple[Tuple[torch.FloatTensor]]] = None,
        doc_scores: Optional[torch.FloatTensor] = None,
        context_input_ids: Optional[torch.LongTensor] = None,
        context_attention_mask: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        output_retrieved: Optional[bool] = None,
        n_docs: Optional[int] = None,
    ) -> Union[Tuple[torch.Tensor], RetrievAugLMOutput]:
        n_docs = n_docs if n_docs is not None else self.config.n_docs
        use_cache = use_cache if use_cache is not None else self.config.use_cache
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        output_retrieved = output_retrieved if output_retrieved is not None else self.config.output_retrieved

        # whether retriever has to be used
        has_to_retrieve = (
            self.retriever is not None
            and (context_input_ids is None or context_attention_mask is None or doc_scores is None)
            and encoder_outputs is None
        )
        # encoder_outputs are pre-computed during RAG-token generation
        if encoder_outputs is None:
            if has_to_retrieve:
                question_enc_outputs = self.question_encoder(
                    input_ids, attention_mask=attention_mask, return_dict=True
                )
                question_encoder_last_hidden_state = question_enc_outputs[0]  # hidden states of question encoder

                retriever_outputs = self.retriever(
                    input_ids,
                    question_encoder_last_hidden_state.cpu().detach().to(torch.float32).numpy(),
                    prefix=self.generator.config.prefix,
                    n_docs=n_docs,
                    return_tensors="pt",
                )[0]

                (
                    context_input_ids,
                    context_attention_mask,
                    retrieved_doc_embeds,
                    retrieved_doc_ids,
                ) = (
                    retriever_outputs["context_input_ids"],
                    retriever_outputs["context_attention_mask"],
                    retriever_outputs["retrieved_doc_embeds"],
                    retriever_outputs["doc_ids"],
                )

                # set to correct device
                retrieved_doc_embeds = retrieved_doc_embeds.to(question_encoder_last_hidden_state)
                context_input_ids = context_input_ids.to(input_ids)
                context_attention_mask = context_attention_mask.to(input_ids)

                # compute doc_scores
                doc_scores = torch.bmm(
                    question_encoder_last_hidden_state.unsqueeze(1),
                    retrieved_doc_embeds.transpose(1, 2),
                ).squeeze(1)
            else:
                assert context_input_ids is not None, (
                    "Make sure that `context_input_ids` are passed, if no `retriever` is set. Alternatively, you can"
                    " set a retriever using the `set_retriever(...)` function."
                )
                assert context_attention_mask is not None, (
                    "Make sure that `context_attention_mask` are passed, if no `retriever` is set. Alternatively, you"
                    " can set a retriever using the `set_retriever(...)` function."
                )
                assert doc_scores is not None, (
                    "Make sure that `doc_scores` are passed, if no `retriever` is set. Alternatively, you can set a"
                    " retriever using the `set_retriever(...)` function."
                )

        assert (
            doc_scores is not None
        ), "Make sure that `doc_scores` are passed when passing `encoder_outputs` to the forward function."

        assert (doc_scores.shape[1] % n_docs) == 0, (
            f" The first dimension of `context_input_ids` should be a multiple of `n_docs`={n_docs}, but is"
            f" {context_input_ids.shape[0]}."
        )

        # Decoder input without context documents
        if decoder_input_ids is not None:
            decoder_input_ids = decoder_input_ids.repeat_interleave(n_docs, dim=0)

        if decoder_attention_mask is not None:
            decoder_attention_mask = decoder_attention_mask.repeat_interleave(n_docs, dim=0)

        gen_outputs = self.generator(
            input_ids=context_input_ids,
            attention_mask=context_attention_mask,
            encoder_outputs=encoder_outputs,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            past_key_values=past_key_values,
            use_cache=use_cache,
            output_attentions=output_attentions,
            return_dict=True,
        )

        if not has_to_retrieve:
            question_encoder_last_hidden_state = None
            question_enc_hidden_states = None
            question_enc_attentions = None
            retrieved_doc_embeds = None
            retrieved_doc_ids = None
        else:
            question_enc_hidden_states = question_enc_outputs.hidden_states
            question_enc_attentions = question_enc_outputs.attentions

        if not has_to_retrieve or not output_retrieved:
            # don't output retrieved docs
            context_input_ids = (None,)
            context_attention_mask = None
            retrieved_doc_embeds = None
            retrieved_doc_ids = None

        return RetrievAugLMOutput(
            logits=gen_outputs.logits,
            doc_scores=doc_scores,
            past_key_values=gen_outputs.past_key_values,
            context_input_ids=context_input_ids,
            context_attention_mask=context_attention_mask,
            retrieved_doc_embeds=retrieved_doc_embeds,
            retrieved_doc_ids=retrieved_doc_ids,
            question_encoder_last_hidden_state=question_encoder_last_hidden_state,
            question_enc_hidden_states=question_enc_hidden_states,
            question_enc_attentions=question_enc_attentions,
            generator_enc_last_hidden_state=gen_outputs.encoder_last_hidden_state,
            generator_enc_hidden_states=gen_outputs.encoder_hidden_states,
            generator_enc_attentions=gen_outputs.encoder_attentions,
            generator_dec_hidden_states=gen_outputs.decoder_hidden_states,
            generator_dec_attentions=gen_outputs.decoder_attentions,
        )

In [5]:
# ConfidentialRagSequenceForGeneration

from concurrent.futures import ThreadPoolExecutor
from typing import Optional
from transformers.configuration_rag import RagConfig
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel
from transformers.modeling_rag import RagSequenceForGeneration
from transformers.retrieval_rag import RagRetriever

import torch


class ConfidentialRagSequenceForGeneration(RagSequenceForGeneration):
    def __init__(
        self,
        config: Optional[PretrainedConfig] = None,
        question_encoder: Optional[PreTrainedModel] = None,
        generator: Optional[PreTrainedModel] = None,
        retriever: Optional[RagRetriever] = None,
        **kwargs,
    ):
        assert config is not None or (
            question_encoder is not None and generator is not None
        ), "Either a configuration or an encoder and a generator has to be provided."

        if config is None:
            config = RagConfig.from_question_encoder_generator_configs(
                question_encoder.config, generator.config, **kwargs
            )
        # super().__init__(config)
        self.normalOutputTopK = 1
        self.numConfidential = 0
        self.generationMode = "normalConfidential"

        # instantiate model
        self.rag = ConfidentialRagModel(
            config=config,
            question_encoder=question_encoder,
            generator=generator,
            retriever=retriever,
        )

    @torch.no_grad()
    def generate(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.LongTensor] = None,
        ctxInputIds: Optional[torch.LongTensor] = None,
        context_attention_mask: Optional[torch.LongTensor] = None,
        doc_scores: Optional[torch.FloatTensor] = None,
        do_deduplication: Optional[bool] = None,  # defaults to True
        num_return_sequences: Optional[int] = None,  # defaults to 1
        num_beams: Optional[int] = None,  # defaults to 1
        n_docs: Optional[int] = None,
        **model_kwargs,
    ) -> torch.LongTensor:
        n_docs = n_docs if n_docs is not None else self.config.n_docs
        n_docs = n_docs - self.numConfidential
        do_deduplication = do_deduplication if do_deduplication is not None else self.config.do_deduplication
        num_doc_return_sequences = (
            num_return_sequences if num_return_sequences is not None else self.config.num_return_sequences
        )
        num_beams = num_beams if num_beams is not None else self.config.num_beams
        assert (
            input_ids is not None or ctxInputIds is not None
        ), " At least one of input_ids or context_input_ids must be given"

        def doNormalConfidentialGeneration():
            if self.retriever is not None:
                question_hidden_states = self.question_encoder(input_ids, attention_mask=attention_mask)[0]
                retrievalResult = self.retriever(
                    input_ids,
                    question_hidden_states.cpu().detach().to(torch.float32).numpy(),
                    prefix=self.generator.config.prefix,
                    n_docs=n_docs,
                    numConfidential=self.numConfidential,
                    return_tensors="pt",
                )

                ctxInputIds = retrievalResult[0]["context_input_ids"]
                confidentialDocs = retrievalResult[1]

                # set to correct device
                ctxInputIds = ctxInputIds.to(input_ids)

            hypos = []
            model_kwargs["num_beams"] = num_beams
            model_kwargs["num_return_sequences"] = num_beams
            model_kwargs["attention_mask"] = None

            batch_size = input_ids.shape[0] if input_ids is not None else ctxInputIds.shape[0] // n_docs

            for index in range(batch_size):
                # MARK: normal generation stage.
                # first, generate beams from documents:
                generator_input_ids = ctxInputIds[index * n_docs : (index + 1) * n_docs]  # (n_docs, max_len)

                output_sequences = self.generator.generate(
                    generator_input_ids,
                    **model_kwargs,
                )  # n_docs * n_beam, tgt_len
                if do_deduplication:
                    # do_deduplication, max_output_len
                    output_sequences = torch.stack(list({str(k.tolist()): k for k in output_sequences}.values()))

                # after deduplication, this number can be less than n_docs*n_beam
                num_candidates = output_sequences.shape[0]

                # then, run model forwards to get nll scores:
                if input_ids is not None:
                    new_input_ids = input_ids[index : index + 1].repeat(num_candidates, 1)
                    outputs = self(new_input_ids, labels=output_sequences, exclude_bos_score=True)
                else:  # input_ids is None, need context_input_ids/mask and doc_scores
                    assert context_attention_mask is not None, (
                        "Make sure that `context_attention_mask` are passed, if no `input_ids` is set. Alternatively, you"
                        " can set a retriever using the `set_retriever(...)` function."
                    )
                    assert doc_scores is not None, (
                        "Make sure that `doc_scores` are passed, if no `input_ids` is set. Alternatively, you can set a"
                        " retriever using the `set_retriever(...)` function."
                    )

                    individual_input_ids = generator_input_ids.repeat(
                        num_candidates, 1
                    )  # (num_candidates*n_docs, max_len)

                    individual_attention_mask = context_attention_mask[index * n_docs : (index + 1) * n_docs]
                    individual_attention_mask = individual_attention_mask.repeat(num_candidates, 1)

                    individual_doc_scores = doc_scores[index : (index + 1), :]  # doc_scores.shape = [batch, n_docs]
                    individual_doc_scores = individual_doc_scores.repeat(num_candidates, 1)  # [num_candidates, n_docs]

                    outputs = self(
                        context_input_ids=individual_input_ids,
                        context_attention_mask=individual_attention_mask,
                        doc_scores=individual_doc_scores,
                        labels=output_sequences,
                        exclude_bos_score=True,
                    )

                normalOutputTopK = self.normalOutputTopK
                top_cand_inds = (-outputs["loss"]).topk(normalOutputTopK)[1]

                # MARK: confidential generation stage.
                confidentialDoc = confidentialDocs[index]
                confidentialRagInputIds = self.retriever.concatConfidentialDocs(
                    inputIds=output_sequences[top_cand_inds],
                    questionIputIds=input_ids[index : index + 1],
                    confidentialDocs=[confidentialDoc for _ in range(normalOutputTopK)],
                    return_tensors="pt",
                )["input_ids"]
                confidentialRagInputIds = confidentialRagInputIds.to(input_ids)
                confidentialOutputSeqence = self.generator.generate(
                    confidentialRagInputIds,
                    **model_kwargs,
                )
                if do_deduplication:
                    # do_deduplication, max_output_len
                    confidentialOutputSeqence = torch.stack(
                        list({str(k.tolist()): k for k in confidentialOutputSeqence}.values())
                    )

                num_candidates = confidentialOutputSeqence.shape[0]
                new_input_ids = input_ids[index : index + 1].repeat(num_candidates, 1)
                finalOutputs = self(
                    new_input_ids,
                    labels=confidentialOutputSeqence,
                    exclude_bos_score=True,
                )
                top_cand_inds = (-finalOutputs["loss"]).topk(num_doc_return_sequences)[1]

                hypos.append(confidentialOutputSeqence[top_cand_inds])

            return self._cat_and_pad(hypos, pad_token_id=self.config.generator.pad_token_id)

        def doParallelSummaryGeneration():
            ctxNormalInputIds = ctxInputIds
            if self.retriever is not None and ctxNormalInputIds is None:
                question_hidden_states = self.question_encoder(input_ids, attention_mask=attention_mask)[0]
                retrievalResult = self.retriever(
                    input_ids,
                    question_hidden_states.cpu().detach().to(torch.float32).numpy(),
                    prefix=self.generator.config.prefix,
                    n_docs=n_docs,
                    numConfidential=self.numConfidential,
                    return_tensors="pt",
                )

                ctxNormalInputIds = retrievalResult[0]["context_input_ids"]

                confidentialDocs = retrievalResult[1]
                ctxConfidentialInputIds = self.retriever.concatConfidentialDocs(
                    inputIds="",
                    questionIputIds=input_ids,
                    confidentialDocs=confidentialDocs,
                    return_tensors="pt",
                )["input_ids"]
                # set to correct device
                ctxNormalInputIds = ctxNormalInputIds.to(input_ids)
                ctxConfidentialInputIds = ctxConfidentialInputIds.to(input_ids)

            hypos = []
            model_kwargs["num_beams"] = num_beams
            model_kwargs["num_return_sequences"] = num_beams
            model_kwargs["attention_mask"] = None

            batch_size = input_ids.shape[0] if input_ids is not None else ctxNormalInputIds.shape[0] // n_docs

            executor = ThreadPoolExecutor(max_workers=3)
            for index in range(batch_size):
                # MARK: PARALLEL STAGE
                def doParallelGenerate(inputIds):
                    def doGenerate(ids):
                        output = self.generator.generate(
                            ids,
                            **model_kwargs,
                        )
                        return output

                    result = executor.submit(doGenerate, inputIds)
                    return result

                normalInputIds = ctxNormalInputIds[index * n_docs : (index + 1) * n_docs]
                confidentialInputIds = ctxConfidentialInputIds[
                    index * self.numConfidential : (index + 1) * self.numConfidential
                ]
                confidentialInputShape = confidentialInputIds.shape
                confidentialInputIds = confidentialInputIds.view(
                    -1, confidentialInputShape[0] * confidentialInputShape[1]
                )
                results = [doParallelGenerate(ids) for ids in [normalInputIds, confidentialInputIds]]
                normalCandidates, confidentialCandidates = (re.result() for re in results)

                # MARK: SUMMARY STAGE
                def combineCandidates(a, b):
                    long, short = (a, b) if a.shape[-1] > b.shape[-1] else (b, a)
                    padid = self.retriever.generator_tokenizer.pad_token_id
                    short = torch.nn.functional.pad(
                        short,
                        (0, long.shape[-1] - short.shape[-1], 0, 0),
                        mode="constant",
                        value=padid,
                    )
                    return torch.cat((long, short))

                candidates = combineCandidates(normalCandidates, confidentialCandidates)
                if do_deduplication:
                    candidates = torch.stack(list({str(k.tolist()): k for k in candidates}.values()))
                num_candidates = candidates.shape[0]
                new_input_ids = input_ids[index : index + 1].repeat(num_candidates, 1)
                outputs = self(new_input_ids, labels=candidates, exclude_bos_score=True)
                top_cand_inds = (-outputs["loss"]).topk(num_doc_return_sequences)[1]
                hypos.append(candidates[top_cand_inds])

            executor.shutdown(wait=True)
            return self._cat_and_pad(hypos, pad_token_id=self.config.generator.pad_token_id)

        if self.generationMode == "normalConfidential":
            return doNormalConfidentialGeneration()
        elif self.generationMode == "parallelSummary":
            return doParallelSummaryGeneration()
        else:  # default is normal-confidential mode.
            return doNormalConfidentialGeneration()

In [6]:
# Dataset
import random


def evaluationDataLoader(ratio=0.1, shuffle=True):
    with open(args.evaluation_set, "r") as eval_file, open(args.gold_data_path, "r") as gold_file:
        questions = eval_file.readlines()
        answers = gold_file.readlines()
        assert len(questions) == len(answers), "Question and Answer size must be equal."
        siz = len(questions)
        qa = [(questions[i], answers[i]) for i in range(siz)]
        if shuffle:
            random.shuffle(qa)
        xy = qa[: int(siz * ratio)]
        x = [item[0] for item in xy]
        y = [item[1] for item in xy]
        return x, y

In [7]:
# retriever init
model_kwargs = {
    "n_docs": args.n_docs,
}

retriever = RagRetriever.from_pretrained(args.model_name_or_path, **model_kwargs)
retriever.init_retrieval()

In [8]:
# model init
model = RagSequenceForGeneration.from_pretrained(args.model_name_or_path, retriever=retriever, **model_kwargs)
model.to(args.device)

In [9]:
# evaluation dataset
X, Y = evaluationDataLoader(0.01, shuffle=False)

In [10]:
# encode input
batchSize = args.eval_batch_size

inputs_dict = model.retriever.question_encoder_tokenizer.batch_encode_plus(
    X[:batchSize], return_tensors="pt", padding=True, truncation=True
)
input_ids = inputs_dict.input_ids.to(args.device)
attention_mask = inputs_dict.attention_mask.to(args.device)

In [11]:
# 1. retrival
question_hidden_states = model.rag.question_encoder(input_ids, attention_mask=attention_mask)[0]
context_input_ids = retriever(
    input_ids,
    question_hidden_states.cpu().detach().to(torch.float32).numpy(),
    prefix=None,
    n_docs=args.n_docs,
    return_tensors="pt",
)["context_input_ids"]
context_input_ids = context_input_ids.to(args.device)

In [12]:
# 2. candidate
kwargs = {}
kwargs["early_stopping"] = False
kwargs["bad_words_ids"] = [[0, 0]]
kwargs["min_length"] = args.min_length
kwargs["max_length"] = args.max_length
kwargs["num_beams"] = args.num_beams
kwargs["num_return_sequences"] = args.num_beams
kwargs["attention_mask"] = None

index = 0
generator_input_ids = context_input_ids[index * args.n_docs : (index + 1) * args.n_docs]  # (n_docs, max_len)
output_sequences = model.generator.generate(
    generator_input_ids,
    **kwargs,
)  # n_docs * n_beam, tgt_len

In [13]:
tmp = model.retriever.generator_tokenizer.batch_decode(output_sequences, skip_special_tokens=True)
print(tmp)

In [151]:
confidentialRagRetriever = ConfidentialRagRetriever(
    config=retriever.config,
    question_encoder_tokenizer=retriever.question_encoder_tokenizer,
    generator_tokenizer=retriever.generator_tokenizer,
    index=retriever.index,
)

In [153]:
# 1. confidential retrival
confidentialRetrieverResult = confidentialRagRetriever(
    input_ids,
    question_hidden_states.cpu().detach().to(torch.float32).numpy(),
    prefix=None,
    n_docs=args.n_docs - args.numConfidential,
    numConfidential=args.numConfidential,
    return_tensors="pt",
)
ctxNormalInputIds = confidentialRetrieverResult[0]["context_input_ids"]
ctxNormalInputIds = ctxNormalInputIds.to(args.device)

# confidential docs
confidentialDocs = confidentialRetrieverResult[1]
ctxConfidentialInputIds = confidentialRagRetriever.concatConfidentialDocs(
    inputIds="",
    questionIputIds=input_ids,
    confidentialDocs=confidentialDocs,
    return_tensors="pt",
)["input_ids"]
ctxConfidentialInputIds = ctxConfidentialInputIds.to(args.device)

In [154]:
index = 0
normalInputIds = ctxNormalInputIds[
    index * (args.n_docs - args.numConfidential) : (index + 1) * (args.n_docs - args.numConfidential)
]
confidentialInputIds = ctxConfidentialInputIds[index * args.numConfidential : (index + 1) * args.numConfidential]
normalCandidates = model.generator.generate(
    normalInputIds,
    **kwargs,
)
confidentialCandidates = model.generator.generate(
    confidentialInputIds,
    **kwargs,
)


def combineCandidates(a, b):
    long, short = (a, b) if a.shape[-1] > b.shape[-1] else (b, a)
    padid = confidentialRagRetriever.generator_tokenizer.pad_token_id
    short = torch.nn.functional.pad(
        short,
        (0, long.shape[-1] - short.shape[-1], 0, 0),
        mode="constant",
        value=padid,
    )
    return torch.cat((long, short))


candidates = combineCandidates(normalCandidates, confidentialCandidates)

In [162]:
tmp2 = model.retriever.generator_tokenizer.batch_decode(candidates, skip_special_tokens=True)
print(tmp2)

In [164]:
tmp2 = sorted(tmp2)
tmp = sorted(tmp)